In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [0]:
# results_df=spark.read\
#     .option("inferSchema", True)\
#         .json("abfss://demofiles@formula1adls.dfs.core.windows.net/source_files/results.json")
# results_df.display()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, FloatType

results_schema = StructType(fields=[StructField("resultId", IntegerType(), False),
                                    StructField("raceId", IntegerType(), True),
                                    StructField("driverId", IntegerType(), True),
                                    StructField("constructorId", IntegerType(), True),
                                    StructField("number", IntegerType(), True),
                                    StructField("grid", IntegerType(), True),
                                    StructField("position", IntegerType(), True),
                                    StructField("positionText", StringType(), True),
                                    StructField("positionOrder", IntegerType(), True),
                                    StructField("points", FloatType(), True),
                                    StructField("laps", IntegerType(), True),
                                    StructField("time", StringType(), True),
                                    StructField("milliseconds", IntegerType(), True),
                                    StructField("fastestLap", IntegerType(), True),
                                    StructField("rank", IntegerType(), True),
                                    StructField("fastestLapTime", StringType(), True),
                                    StructField("fastestLapSpeed", FloatType(), True),
                                    StructField("statusId", StringType(), True)])


In [0]:
results_df=spark.read\
    .schema(results_schema)\
        .json("/Volumes/formula1_dev/bronze/demo_files/results.json")
        # .json("abfss://demofiles@formula1adls.dfs.core.windows.net/source_files/results.json")
        # /Volumes/formula1_dev/bronze/demo_files/results.json


In [0]:
results_df.display()

In [0]:
results_df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp,current_date
v= results_df.withColumnRenamed("resultId", "result_id")\
            .withColumnRenamed("raceId", "race_id")\
            .withColumnRenamed("driverId", "driver_id")\
            .withColumnRenamed("constructorId", "constructor_id")\
            .withColumnRenamed("positionText", "position_text")\
            .withColumnRenamed("positionOrder", "position_order")\
            .withColumnRenamed("fastestLap", "fastest_lap")\
            .withColumnRenamed("fastestLapTime", "fastest_lap_time")\
            .withColumnRenamed("fastestLapSpeed", "fastest_lap_speed")\
            .withColumn("ingestion_timestamp", current_timestamp())\
            .withColumn("ingestion_date", current_date())\
            .drop("url")


In [0]:
v.count()

In [0]:
v.write.mode("append").format("delta").option("mergeSchema", "true").option("path", "abfss://raw@formula1adls.dfs.core.windows.net/results").saveAsTable(f"formula1_{env}.bronze.results")


In [0]:
df=spark.table(f"formula1_{env}.bronze.results")
df.count()
# df.limit(10).display()

In [0]:
df=spark.table(f"formula1_{env}.bronze.results")
df.count()
# df.limit(10).display()

In [0]:
# 24960+20
# day 1 --> file -->20
# day2 -> file1-->24960